
# 練習問題: SIMD + マルチコアで台数効果を出す

## 目標

SIMD (ベクトル型) によって1コア内で性能を出しているコードに `#pragma omp parallel for` (Fortran では `!$omp parallel do`) を1つ追加するだけで, 外側のループがマルチコア並列化され, スレッド数に応じてGFLOPS値が伸びることを体験する.

## 課題

`simd_multicore.cpp` (または `simd_multicore.f90`) は, 互いに独立な漸化式 `t = a*t + b` を多数 (`m` 本) 計算するプログラムである.

- C++版は, ベクトル型 `doublev` (`double` を `nl` 個束ねた型) を使い, `nl` 本の漸化式を1命令でまとめて進めることでSIMD化されている. しかし外側のループ (`i += nl`) はまだ1スレッドでしか動かない.
- Fortran版は, 内側の漸化式を `!$omp simd` でSIMD化してあるが, やはり外側のループは1スレッドである.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, 外側のループをマルチコア並列化せよ.

- C++: `for (long i = 0; i < m; i += nl)` の直前に `#pragma omp parallel for` を1行加える.
- Fortran: `do i = 1, m` の直前に `!$omp parallel do` を1行加える.

それ以外のコードを変更する必要はない.

なお C++版の先頭には `enum { nl = 8 };` というベクトル長の定義がある. 課題が動いたら, `nl` を 8, 16, 32 (2のべき乗) と変えて性能がどう変わるかも試してみよ.

<font color="red">注:</font> ベクトル型 (`vector_size`) はC/C++独自の拡張であり, Fortranには相当する機能が無い. そのためFortran版ではSIMD化を `!$omp simd` (内側) に, マルチコア並列化を `!$omp parallel do` (外側) に委ねている.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore simd_multicore.cpp -o simd_multicore.exe

# Fortran
nvfortran -fast -mp=multicore simd_multicore.f90 -o simd_multicore.exe
```

`OMP_NUM_THREADS` を変えながら実行する. `OMP_PROC_BIND=true` は各スレッドを特定のコアに固定し, 台数効果の測定を安定させる指示である. `m` はスレッド数に比例させて与える (各スレッドの仕事量を一定に保つ).

```
OMP_PROC_BIND=true OMP_NUM_THREADS=1 ./simd_multicore.exe 64 $((100 * 1000 * 1000))
```

ジョブとして投入する場合の例 (`#PJM` の指定):

```
#PJM -L rscgrp=lecture-a

for th in 1 2 4 8 ; do
    OMP_PROC_BIND=true OMP_NUM_THREADS=${th} ./simd_multicore.exe $((64 * ${th})) $((100 * 1000 * 1000)) | grep GFLOPS
done
```

## 期待される結果

`#pragma omp parallel for` (`!$omp parallel do`) を追加すると, `OMP_NUM_THREADS` を増やすにつれてGFLOPS値がほぼスレッド数に比例して伸びる.
追加する前は (1スレッドでしか動かないため) スレッド数を増やしてもGFLOPS値が変わらないことと比べてみよ.
最後に `OK` と表示されれば計算結果は正しい.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ simd_multicore.cpp
#include <cstdio>
#include <cstdlib>
#include <cassert>
#include <cmath>
#include <omp.h>

/* ベクトル長 nl. 2 のべき乗 (8, 16, 32, ...) に変えて性能を比べてみよ */
enum { nl = 8 };

typedef double doublev __attribute__((vector_size(sizeof(double) * nl),
                                      aligned(sizeof(double))));

/* スカラ u を全要素 u のベクトルに */
doublev uniform(double u) {
  doublev v;
  for (long i = 0; i < nl; i++) {
    v[i] = u;
  }
  return v;
}

/* u, u+1, u+2, ... からなるベクトル */
doublev range(double u) {
  doublev v;
  for (long i = 0; i < nl; i++) {
    v[i] = u + i;
  }
  return v;
}

/* ベクトル型 v を配列 a の先頭 nl 要素に書き込む */
void storev(double * a, doublev v) {
  for (int i = 0; i < nl; i++) {
    a[i] = v[i];
  }
}

/* b, t をベクトル型にした lin_rec.
   nl 個の独立な漸化式を同時に進める */
doublev lin_rec(double a, doublev b, double c, long n) {
  doublev t = uniform(c);
  for (long j = 0; j < n; j++) {
    t = a * t + b;
  }
  return t;
}

int main(int argc, char ** argv) {
  long m     = (1 < argc ? atol(argv[1]) : 8);
  long n     = (2 < argc ? atol(argv[2]) : 1000 * 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);
  printf("m = %ld, n = %ld, nl = %d\n", m, n, nl);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体 (nl 回ずつまとめて SIMD 実行). */
  // TODO: 下の for ループの直前に #pragma omp parallel for を1行追加し, 外側のループ (互いに独立) をマルチコアで並列化せよ.
  for (long i = 0; i < m; i += nl) {
    storev(&x[i], lin_rec(0.99, range(i) + 1, 1.0, n));
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */

  /* 答え表示 (x[i] = 100 * (i + 1) くらいのはず) */
  long err = 0;
  for (long i = 0; i < m; i++) {
    if (fabs(x[i] - 100 * (i + 1)) > 1.0e-3) {
      printf("x[%3ld] = %9.3f\n", i, x[i]);
      err++;
    }
  }
  if (err == 0) {
    printf("OK\n");
  }
  double flops = 2. * (double)m * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore simd_multicore.cpp -o simd_multicore_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./simd_multicore_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ simd_multicore.f90
! Fortran には C/C++ のベクトル型拡張 (vector_size) に相当する機能が無い.
! そこで, 内側の漸化式ループを !$omp simd で SIMD 化し,
! 外側の独立な要素のループを !$omp parallel do でマルチコア並列化して,
! SIMD + 命令レベル並列 + マルチコアをコンパイラに任せる.

module lin_rec_mod
contains
  ! t = a*t + b を n 回繰り返す
  real(8) function lin_rec(a, b, c, n) result(t)
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    integer(8) :: j
    t = c
    !$omp simd
    do j = 1, n
       t = a * t + b
    end do
  end function lin_rec
end module lin_rec_mod

program simd_multicore
  use lin_rec_mod
  use omp_lib
  implicit none
  integer(8) :: m, n, i, err
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  character(len=32) :: arg

  m = 8
  n = 1000_8 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read(arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read(arg, *) n
  end if
  allocate(x(m))
  print '(A,I0,A,I0)', "m = ", m, ", n = ", n

  t0 = omp_get_wtime()
  ! TODO: 下の do ループの直前に !$omp parallel do を1行追加し, 外側 (互いに独立な要素) のループをマルチコアで並列化せよ.
  do i = 1, m
     x(i) = lin_rec(0.99d0, dble(i), 1.0d0, n)
  end do
  ! TODO: 上の do ループに対応する !$omp end parallel do を書け.
  t1 = omp_get_wtime()
  dt = t1 - t0

  err = 0
  do i = 1, m
     if (abs(x(i) - 100.0d0 * dble(i)) > 1.0d-3) then
        print '(A,I3,A,F9.3)', "x(", i, ") = ", x(i)
        err = err + 1
     end if
  end do
  if (err == 0) print '(A)', "OK"

  flops = 2.0d0 * dble(m) * dble(n)
  print '(A,F7.3,A)', "elapsed    : ", dt, "  sec"
  print '(A,ES9.2)', "flops      : ", flops
  print '(F8.3,A)', flops / dt * 1d-9, " GFLOPS"
end program simd_multicore

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore simd_multicore.f90 -o simd_multicore_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./simd_multicore_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:simd_multicore.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:simd_multicore.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:simd_multicore.cpp}